<a href="https://colab.research.google.com/github/samerabouchakra88-spec/samer-repo-1/blob/main/adding%20Feature%20engineered.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

# ============================================================================
# STEP 1: LOAD THE DATASET
# ============================================================================
# For Google Colab, upload your CSV file or mount Google Drive:
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv('ammatourolives_final.csv')

# For local testing:
df = pd.read_csv('ammatourolives_final.csv')

print("Original dataset shape:", df.shape)
print("Original columns:", df.columns.tolist())

# ============================================================================
# STEP 2: CATEGORICAL ENCODING
# ============================================================================
print("\n[1/8] Encoding categorical variables...")

# Create mapping dictionaries for categorical variables
soil_type_mapping = {'Clay': 0, 'Loam': 1, 'Sandy_Loam': 2}
pruning_intensity_mapping = {'Light': 1, 'Moderate': 2, 'Heavy': 0}

df['Soil_Type_enc'] = df['Soil_Type'].map(soil_type_mapping)
df['Pruning_Intensity_enc'] = df['Pruning_Intensity'].map(pruning_intensity_mapping)

# ============================================================================
# STEP 3: QUANTILE BINNING FOR OLIVES PRODUCTION
# ============================================================================
print("[2/8] Creating quantile bins for Olives_Produced_kg_per_hectare...")

# Create 10 quantile bins for olive production
df['Olives_Produced_kg_per_hectare_quantile_bins'] = pd.qcut(
    df['Olives_Produced_kg_per_hectare'],
    q=10,
    duplicates='drop'
)

# ============================================================================
# STEP 4: LAGGED FEATURES (PREVIOUS YEAR VALUES)
# ============================================================================
print("[3/8] Creating lagged features (previous year values)...")

# Sort by Field_ID and Year to ensure correct lagging
df = df.sort_values(['Field_ID', 'Year']).reset_index(drop=True)

# Create lagged features for each field
df['Prev_Year_Oil'] = df.groupby('Field_ID')['Oil_Yield_kg_per_hectare'].shift(1)
df['Prev_Year_Olives'] = df.groupby('Field_ID')['Olives_Produced_kg_per_hectare'].shift(1)

# ============================================================================
# STEP 5: YEAR-OVER-YEAR CHANGE
# ============================================================================
print("[4/8] Calculating year-over-year changes...")

# Calculate YoY change in oil yield
df['Oil_YoY_Change'] = (df['Oil_Yield_kg_per_hectare'] - df['Prev_Year_Oil']) / df['Prev_Year_Oil']

# ============================================================================
# STEP 6: ROLLING AVERAGES (3-YEAR AND 2-YEAR)
# ============================================================================
print("[5/8] Computing rolling averages...")

# 3-year rolling average for rainfall
df['Rain_3yr_avg'] = df.groupby('Field_ID')['Annual_Rainfall_mm_year'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

# 2-year rolling average for rainfall
df['Rain_2yr_avg'] = df.groupby('Field_ID')['Annual_Rainfall_mm_year'].transform(
    lambda x: x.rolling(window=2, min_periods=1).mean()
)

# 3-year rolling average for irrigation
df['Irr_3yr_avg'] = df.groupby('Field_ID')['Irrigation_L_per_tree'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

# ============================================================================
# STEP 7: COMPOSITE FEATURES
# ============================================================================
print("[6/8] Creating composite features...")

# Heat Stress: Frost Days × Altitude × 0.044 (scaling factor)
df['Heat_Stress'] = df['Frost_Days_count'] * df['Altitude_m'] * 0.044

# Temperature-Rainfall Index: (Summer Max Temp × 100) / Annual Rainfall
df['Temp_Rain_Index'] = (df['Summer_Max_Temp_C_avg'] * 100) / df['Annual_Rainfall_mm_year']

# Frost-Altitude interaction: (Frost Days × Altitude) / 100
df['Frost_Altitude'] = (df['Frost_Days_count'] * df['Altitude_m']) / 100

# Spring Rainfall Ratio: Spring Rainfall / Annual Rainfall
df['Spring_Rain_Ratio'] = df['Spring_Rainfall_mm_season'] / df['Annual_Rainfall_mm_year']

# Total Water: Annual Rainfall + Irrigation
df['Total_Water'] = df['Annual_Rainfall_mm_year'] + df['Irrigation_L_per_tree']

# Water per Tree: Total Water / Tree Density
df['Water_per_Tree'] = df['Total_Water'] / df['Tree_Density_per_hectare']

# Irrigation-Rainfall Ratio: Irrigation / Annual Rainfall
df['Irr_Rain_Ratio'] = df['Irrigation_L_per_tree'] / df['Annual_Rainfall_mm_year']

# Tree Productivity: Note - This feature appears to be derived from a custom formula
# that is not directly reproducible from the raw features. In the original dataset,
# it shows values like 10759, 10962, etc. which don't match standard combinations.
# For now, we'll use Olives / Density as the closest approximation.
# If you have the exact formula, please update this line.
df['Tree_Productivity'] = df['Olives_Produced_kg_per_hectare'] / df['Tree_Density_per_hectare']

# Fertilizer per Tree: Fertilizer × Tree Density
df['Fert_per_Tree'] = df['Fertilizer_20_20_20_kg_per_tree'] * df['Tree_Density_per_hectare']

# Age-Density Ratio: Tree Age / Tree Density
df['Age_Density_Ratio'] = df['Tree_Age_Years'] / df['Tree_Density_per_hectare']

# ============================================================================
# STEP 8: FIELD-LEVEL AGGREGATION
# ============================================================================
print("[7/8] Computing field-level aggregations...")

# Calculate mean oil yield for each field
field_oil_mean = df.groupby('Field_ID')['Oil_Yield_kg_per_hectare'].transform('mean')
df['Field_Oil_Mean'] = field_oil_mean

# Oil vs Field Mean: Current Oil Yield / Field Mean Oil Yield
df['Oil_vs_Field_Mean'] = df['Oil_Yield_kg_per_hectare'] / df['Field_Oil_Mean']

# ============================================================================
# STEP 9: REORDER COLUMNS TO MATCH ENGINEERED DATASET
# ============================================================================
print("[8/8] Finalizing dataset...")

# Define the final column order
final_columns = [
    'Year', 'Field_ID', 'Altitude_m', 'Annual_Rainfall_mm_year',
    'Spring_Rainfall_mm_season', 'Summer_Max_Temp_C_avg', 'Frost_Days_count',
    'Irrigation_L_per_tree', 'Tree_Age_Years', 'Tree_Density_per_hectare',
    'Fertilizer_20_20_20_kg_per_tree', 'Soil_Type', 'Pest_Pressure_Index',
    'Pruning_Intensity', 'Oil_Yield_kg_per_hectare', 'Olives_Produced_kg_per_hectare',
    'Olives_Produced_kg_per_hectare_quantile_bins', 'Soil_Type_enc',
    'Pruning_Intensity_enc', 'Prev_Year_Oil', 'Prev_Year_Olives', 'Oil_YoY_Change',
    'Rain_3yr_avg', 'Rain_2yr_avg', 'Irr_3yr_avg', 'Heat_Stress', 'Temp_Rain_Index',
    'Frost_Altitude', 'Spring_Rain_Ratio', 'Total_Water', 'Water_per_Tree',
    'Irr_Rain_Ratio', 'Tree_Productivity', 'Fert_per_Tree', 'Age_Density_Ratio',
    'Field_Oil_Mean', 'Oil_vs_Field_Mean'
]

# Reorder columns
df = df[final_columns]

# ============================================================================
# STEP 10: SAVE THE ENGINEERED DATASET
# ============================================================================
print("\nDataset shape after feature engineering:", df.shape)
print("Final columns:", df.columns.tolist())

# Save to CSV
output_filename = 'ammatourolives_engineered.csv'
df.to_csv(output_filename, index=False)
print(f"\n✓ Engineered dataset saved to '{output_filename}'")

# Display sample of engineered dataset
print("\nSample of engineered dataset (first 5 rows, selected columns):")
print(df[['Year', 'Field_ID', 'Oil_Yield_kg_per_hectare', 'Soil_Type_enc',
          'Pruning_Intensity_enc', 'Heat_Stress', 'Temp_Rain_Index',
          'Tree_Productivity', 'Oil_vs_Field_Mean']].head())

# Display data types
print("\nData types of engineered features:")
engineered_cols = [
    'Soil_Type_enc', 'Pruning_Intensity_enc', 'Prev_Year_Oil', 'Oil_YoY_Change',
    'Rain_3yr_avg', 'Heat_Stress', 'Temp_Rain_Index', 'Tree_Productivity',
    'Oil_vs_Field_Mean'
]
for col in engineered_cols:
    print(f"  {col}: {df[col].dtype}")

# Display summary statistics
print("\nSummary statistics for key engineered features:")
print(df[['Heat_Stress', 'Temp_Rain_Index', 'Tree_Productivity',
          'Oil_vs_Field_Mean', 'Water_per_Tree']].describe())

Original dataset shape: (200, 16)
Original columns: ['Year', 'Field_ID', 'Altitude_m', 'Annual_Rainfall_mm_year', 'Spring_Rainfall_mm_season', 'Summer_Max_Temp_C_avg', 'Frost_Days_count', 'Irrigation_L_per_tree', 'Tree_Age_Years', 'Tree_Density_per_hectare', 'Fertilizer_20_20_20_kg_per_tree', 'Soil_Type', 'Pest_Pressure_Index', 'Pruning_Intensity', 'Oil_Yield_kg_per_hectare', 'Olives_Produced_kg_per_hectare']

[1/8] Encoding categorical variables...
[2/8] Creating quantile bins for Olives_Produced_kg_per_hectare...
[3/8] Creating lagged features (previous year values)...
[4/8] Calculating year-over-year changes...
[5/8] Computing rolling averages...
[6/8] Creating composite features...
[7/8] Computing field-level aggregations...
[8/8] Finalizing dataset...

Dataset shape after feature engineering: (200, 37)
Final columns: ['Year', 'Field_ID', 'Altitude_m', 'Annual_Rainfall_mm_year', 'Spring_Rainfall_mm_season', 'Summer_Max_Temp_C_avg', 'Frost_Days_count', 'Irrigation_L_per_tree', 'Tree

In [4]:
df_engineered=pd.read_csv('/content/ammatourolives_engineered.csv')

In [5]:
df_engineered.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 37 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   Year                                          200 non-null    int64  
 1   Field_ID                                      200 non-null    object 
 2   Altitude_m                                    200 non-null    float64
 3   Annual_Rainfall_mm_year                       200 non-null    float64
 4   Spring_Rainfall_mm_season                     200 non-null    float64
 5   Summer_Max_Temp_C_avg                         200 non-null    float64
 6   Frost_Days_count                              200 non-null    float64
 7   Irrigation_L_per_tree                         200 non-null    float64
 8   Tree_Age_Years                                200 non-null    int64  
 9   Tree_Density_per_hectare                      200 non-null    int

In [8]:
df_engineered.isnull().sum()

,0
Year,0
Field_ID,0
Altitude_m,0
Annual_Rainfall_mm_year,0
Spring_Rainfall_mm_season,0
Summer_Max_Temp_C_avg,0
Frost_Days_count,0
Irrigation_L_per_tree,0
Tree_Age_Years,0
Tree_Density_per_hectare,0
